# MiniLM baseline

Encode each comment with MiniLM and average the vectors for each user.
Then predict the five Big Five scores with Ridge using the saved folds.

Install `sentence-transformers` in your notebook environment before running.
The first run needs access to Hugging Face to download the model.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

DATA = Path("../data/preprocessed")  # From the project's src folder
OUTPUT = Path("../data/minilm_baseline")
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64

TRAITS = ["agreeableness", "openness", "conscientiousness",
          "extraversion", "neuroticism"]
ALPHAS = [0.01, 0.1, 1, 10, 100]

users = pd.read_csv(DATA / "main/users.csv").set_index("author")
outer = pd.read_csv(DATA / "outer_folds.csv").set_index("author")
inner = pd.read_csv(DATA / "inner_folds.csv")
assert users.index.is_unique and outer.index.is_unique
assert set(users.index) == set(outer.index)

OUTPUT.mkdir(parents=True, exist_ok=True)

/cluster/home/julieawo/.conda/envs/tdt13idun/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Encode the comments

MiniLM is pretrained and stays frozen. Each comment is encoded independently,
so this step can run before cross-validation. Long comments are truncated by
the model. We keep a running sum for each user to avoid saving millions of
comment vectors.

Run this cell once per dataset and model. You can skip it on later runs if
the saved embeddings were made with the same data and model.

In [2]:
encoder = SentenceTransformer(MODEL)
print("Device:", encoder.device)

dimensions = encoder.get_sentence_embedding_dimension()
sums = np.zeros((len(users), dimensions), dtype=np.float64)
counts = np.zeros(len(users), dtype=int)

chunks = pd.read_csv(DATA / "main/comments.csv", usecols=["author", "body"],
                     keep_default_na=False, chunksize=10_000)

for step, chunk in enumerate(chunks, start=1):
    user_rows = users.index.get_indexer(chunk["author"])
    assert (user_rows >= 0).all()

    vectors = encoder.encode(chunk["body"].tolist(), batch_size=BATCH_SIZE,
                             show_progress_bar=False)
    np.add.at(sums, user_rows, vectors)
    np.add.at(counts, user_rows, 1)

    if step % 10 == 0:
        print(f"Encoded {counts.sum():,} comments")

assert (counts > 0).all()
assert np.array_equal(counts, users["comment_count"].to_numpy())

averages = sums / counts[:, None]
embeddings = pd.DataFrame(averages, index=users.index)
embeddings.to_csv(OUTPUT / "user_embeddings.csv")
print(f"Saved embeddings for {len(embeddings):,} users")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/103 [00:00<00:34,  2.95it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 299.36it/s]

Device: cuda:0


/tmp/ipykernel_314589/1121001812.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimensions = encoder.get_sentence_embedding_dimension()


Encoded 100,000 comments


... (19 outputs left out)

Encoded 2,100,000 comments


Encoded 2,200,000 comments


Encoded 2,300,000 comments


Saved embeddings for 1,311 users


## 2. Train Ridge

Pick alpha for each trait with the inner folds. Then fit Ridge on the full
outer training set and predict scores for the outer test users.

In [3]:
embeddings = pd.read_csv(OUTPUT / "user_embeddings.csv").set_index("author")
assert embeddings.index.is_unique and set(embeddings.index) == set(users.index)
X = embeddings.loc[users.index].to_numpy()
data = users[TRAITS].join(outer)

predictions = []
for fold in sorted(data["outer_fold"].unique()):
    train = data["outer_fold"].ne(fold)
    test = data["outer_fold"].eq(fold)

    inner_fold = inner[inner["outer_fold"] == fold].set_index("author")["inner_fold"]
    assert inner_fold.index.is_unique and set(inner_fold.index) == set(data.index[train])
    inner_fold = inner_fold.reindex(data.index[train]).to_numpy()

    cv = []
    for val_fold in sorted(set(inner_fold)):
        train_idx = (inner_fold != val_fold).nonzero()[0]
        val_idx = (inner_fold == val_fold).nonzero()[0]
        cv.append((train_idx, val_idx))

    for trait in TRAITS:
        target = data[trait]
        search = GridSearchCV(Ridge(solver="lsqr"), {"alpha": ALPHAS}, cv=cv,
                              scoring="neg_root_mean_squared_error")
        search.fit(X[train], target[train])
        predictions.append(pd.DataFrame({
            "author": data.index[test],
            "outer_fold": fold,
            "trait": trait,
            "true": target[test].to_numpy(),
            "minilm_prediction": search.predict(X[test]),
            "mean_prediction": target[train].mean(),
            "alpha": search.best_params_["alpha"],
        }))

    print("Finished fold", fold)

Finished fold 0


Finished fold 1


Finished fold 2


Finished fold 3


Finished fold 4


## 3. Check the results

Each user has one test prediction per trait. Compute the metrics using those
predictions. Pearson correlation is left blank if the predictions are constant.

In [4]:
results = pd.concat(predictions, ignore_index=True)
assert len(results) == len(users) * len(TRAITS)
assert not results.duplicated(["author", "trait"]).any()

metrics = []
for trait, rows in results.groupby("trait"):
    actual = rows["true"]
    for model in ["minilm", "mean"]:
        predicted = rows[f"{model}_prediction"]
        correlation = float("nan")
        if predicted.nunique() > 1 and actual.nunique() > 1:
            correlation = pearsonr(actual, predicted)[0]
        metrics.append({
            "trait": trait,
            "model": model,
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "pearson_r": correlation,
            "r2": r2_score(actual, predicted),
        })

metrics = pd.DataFrame(metrics)
results.to_csv(OUTPUT / "predictions.csv", index=False)
metrics.to_csv(OUTPUT / "metrics.csv", index=False)
display(metrics.round(3))

,trait,model,rmse,pearson_r,r2
0,agreeableness,minilm,30.160,0.185,0.034
1,agreeableness,mean,30.713,-0.054,-0.002
2,conscientiousness,minilm,29.772,0.152,0.023
3,conscientiousness,mean,30.141,-0.050,-0.001
4,extraversion,minilm,28.997,0.279,0.071
5,extraversion,mean,30.129,-0.068,-0.003
6,neuroticism,minilm,31.577,0.186,0.034
7,neuroticism,mean,32.164,-0.061,-0.002
8,openness,minilm,27.439,0.151,0.023
9,openness,mean,27.778,-0.049,-0.001
